In [58]:
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"])
)

In [59]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal['Positive', 'Negative']

llm_structured_output=llm.with_structured_output(llm_schema)

In [60]:
print(llm_structured_output.invoke("The movie was a thrilling adventure with stunning visuals and a captivating storyline."))
print(llm_structured_output.invoke("The movie was a complete disaster, with poor acting and a confusing plot."))

movie_summary_flag='Positive'
movie_summary_flag='Negative'


## Chain with Conditions

In [61]:
#Task 1
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer. Summarize the following text in 2-3 sentences."),
    ("user", "Please summarize the following movie: {text}"),
])

In [62]:
#Task 2
llm_structured_output=llm.with_structured_output(llm_schema)

In [63]:
#Task 3
from langchain_core.runnables import RunnableLambda

def pydantic_json(text: llm_schema) -> str:
    return text.model_dump()["movie_summary_flag"]

dict_maker_runnable = RunnableLambda(pydantic_json)

## Conditional Chain 1

In [64]:
from langchain_core.output_parsers import StrOutputParser

#Task 1
linkedin_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator. Generate a LinkedIn post based on the following text."),
    ("user", "Please generate a LinkedIn post for the following text: {text}"),
])

#Task 2

llm_openai_linkedin = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
    temperature=0.7
)

#Task 3
linkedin_str_parser = StrOutputParser()

#Chain for LinkedIn post generation
chain_linkedin = linkedin_prompt_template | llm_openai_linkedin | linkedin_str_parser

## Conditional Chain 2

In [65]:
def instagram_chain(text: dict):

    text = text["text"]

    #Task 1
    instagram_prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator. Generate an Instagram post based on the following text."),
        ("user", "Please generate an Instagram post for the following text: {text}"),
    ])

    #Task 2
    llm_openai_instagram = ChatOpenAI(
        model="gpt-4o-mini",
        base_url="https://openrouter.ai/api/v1",
        api_key=SecretStr(os.environ["OPENROUTER_API_KEY"]),
        temperature=0.7
    )

    #Task 3
    instagram_str_parser = StrOutputParser()

    #Chain for Instagram post generation
    chain_instagram = instagram_prompt_template | llm_openai_instagram | instagram_str_parser

    chain_instagram_output = chain_instagram.invoke({"text": text})

    return chain_instagram_output

instagram_chain_runnable = RunnableLambda(instagram_chain)

## Final Orchestration

In [66]:
from langchain_core.runnables import RunnableBranch, RunnableParallel

conditional_chain = RunnableBranch(
    (lambda x: "Positive" in x, instagram_chain_runnable),
    chain_linkedin
)


In [67]:
def get_movie_flag(result: llm_schema) -> Literal["Positive", "Negative"]:
    return result.movie_summary_flag

In [68]:
from langchain_core.runnables import RunnableSequence

classification_chain = RunnableSequence(
    prompt_template,
    llm_structured_output,
    RunnableLambda(lambda result: result.movie_summary_flag)
)

In [69]:

from typing import Mapping

def extract_text(value: Mapping[str, str]) -> str:
    return value["text"]

final_orchestration_chain = (
    RunnableParallel(
        flag=classification_chain,
        text=RunnableLambda(extract_text)
    )
    | conditional_chain
)

In [70]:
final_orchestration_chain.invoke({"text": "The movie was a thrilling adventure with stunning visuals and a captivating storyline."})

"🎬✨ Just got back from watching an incredible movie that truly took my breath away! 🚀 The thrilling adventure unfolded with stunning visuals that transported me to another world. The captivating storyline kept me on the edge of my seat from start to finish. \n\nIt's amazing how film can inspire and ignite our creativity. It reminds me of the power of storytelling in our professional lives as well. Just like a great movie, successful projects require a compelling narrative, a strong vision, and the ability to engage and inspire those around us. \n\nWhat was the last movie that left you inspired? Let’s share our favorites! 🎥🍿 #FilmReview #Storytelling #Inspiration #Creativity #MovieNight"